### Bibliotecas

In [1]:
import geopandas as gpd
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

### Importa Shapefiles

In [ ]:
# Função para ler arquivos .shp de uma pasta específica
def importar_shapefiles(caminho_base, prefixo='', sufixo='.shp'):
    geodfs = []
    for raiz, _, arquivos in os.walk(caminho_base):
        for arquivo in arquivos:
            if arquivo.endswith(sufixo) and (prefixo == '' or arquivo.startswith(prefixo)):
                caminho_completo = os.path.join(raiz, arquivo)
                try:
                    gdf = gpd.read_file(caminho_completo)
                    geodfs.append(gdf)
                except Exception as e:
                    pass  # Silenciar erros para evitar interrupções
    return geodfs

# Função para concatenar múltiplos GeoDataFrames em um único
def concatenar_geodfs(geodfs):
    if geodfs:
        return gpd.GeoDataFrame(pd.concat(geodfs, ignore_index=True))
    return gpd.GeoDataFrame()

# Função para importar e concatenar arquivos de uma pasta com opção de paralelização
def importar_e_concatenar_paralelo(caminho_base, prefixo='', sufixo='.shp'):
    with ThreadPoolExecutor() as executor:
        geodfs = list(executor.map(lambda caminho: importar_shapefiles(caminho, prefixo, sufixo), [caminho_base]))
    return concatenar_geodfs(geodfs[0])

# Importação e concatenação dos GeoDataFrames
caminhos = {
    'lotes': r'D:\Drive\Meu Drive\Codigos bee6\bee6_arquivos\solo_inteligente\Raw\Lotes',
    'mapa1': r'D:\Drive\Meu DriveCodigos bee6\bee6_arquivos\solo_inteligente\Raw\Mapa-1-SHP',
    'risco_geologico': r'D:\Drive\Meu DriveCodigos bee6\bee6_arquivos\solo_inteligente\Raw\risco geologico atual',
    'manancial': r'D:\DriveMeu Drive\Codigos bee6\bee6_arquivos\solo_inteligente\Raw\SIRGAS_SHP_manancialreestruturado',
    'mancha_inundacao': r'D:\Drive\Meu DriveCodigos bee6\bee6_arquivos\solo_inteligente\Raw\SIRGAS_SHP_mancha_inundacao',
    'zoneamento': r'D:\Drive\Meu DriveCodigos bee6\bee6_arquivos\solo_inteligente\Raw\zoneamento'}

lote = importar_e_concatenar_paralelo(caminhos['lotes'])
mapa1 = importar_e_concatenar_paralelo(caminhos['mapa1'])
risco_geologico = importar_e_concatenar_paralelo(caminhos['risco_geologico'], prefixo='SIRGAS_SHP_riscogeologicoatual_')
manancial_completo = importar_e_concatenar_paralelo(caminhos['manancial'], prefixo='SIRGAS_SHP_manancialreestruturado_polygon')
mancha_inundacao = importar_e_concatenar_paralelo(caminhos['mancha_inundacao'], prefixo='SIRGAS_SHP_mancha_inundacao_')
zoneamento_completo = importar_e_concatenar_paralelo(caminhos['zoneamento'])

# Importar o arquivo .gpkg de uso predominante
caminho_gpkg = r'D:\Drive\Meu DriveCodigos bee6\bee6_arquivos\solo_inteligente\Raw\SIRGAS_GPKG_uso_predominante_2021.gpkg'
try:
    uso_predominante_gdf = gpd.read_file(caminho_gpkg)
except Exception as e:
    uso_predominante_gdf = gpd.GeoDataFrame()  
    
# Importar o arquivo IPTU CSV
caminho_iptu_csv = r'D:\Drive\Meu Drive\Codigos bee6\bee6_arquivos\bee6_geral\Raw\Download Iptus\IPTU_2024.csv'
iptu = pd.read_csv(caminho_iptu_csv, encoding='latin1', delimiter=';')

### Agrega Iptu e Lote

In [ ]:
# Verificando se as colunas existem antes de tentar manipulá-las

# Tratamento do dataframe `lote`
if all(col in lote.columns for col in ['lo_setor', 'lo_quadra', 'lo_lote']):
    lote['sql'] = lote['lo_setor'].fillna('') + lote['lo_quadra'].fillna('') + lote['lo_lote'].fillna('')
    colunas_para_remover = ['lo_setor', 'lo_quadra', 'lo_lote', 'lo_condomi']
    lote = lote.drop(columns=[col for col in colunas_para_remover if col in lote.columns])
    lote = lote.rename(columns={'lo_tp_quad': 'tipo_quadra', 'lo_tp_lote': 'tipo_lote'})

# Tratamento do dataframe `iptu`
if 'NUMERO DO CONTRIBUINTE' in iptu.columns:
    iptu['sql'] = iptu['NUMERO DO CONTRIBUINTE'].astype(str).str[:-2]

    colunas_para_remover = ['NUMERO DO CONTRIBUINTE', 'ANO DO EXERCICIO', 'NUMERO DA NL', 'DATA DO CADASTRAMENTO', 'NUMERO DO CONDOMINIO']
    iptu = iptu.drop(columns=[col for col in colunas_para_remover if col in iptu.columns])

    iptu.columns = [col.lower() for col in iptu.columns]  # Renomeando colunas para letras minúsculas

# Agregação dos dataframes `lote` e `iptu`
lotes_iptu = iptu.merge(lote, how='left', on='sql') if 'sql' in iptu.columns and 'sql' in lote.columns else iptu

# Conversão e tratamento da coluna 'quantidade de pavimentos'
if 'quantidade de pavimentos' in lotes_iptu.columns:
    lotes_iptu['quantidade de pavimentos'] = pd.to_numeric(lotes_iptu['quantidade de pavimentos'], errors='coerce')
    lotes_iptu['quantidade de pavimentos'].fillna(0, inplace=True)

# Tratamento da coluna 'testada para calculo'
if 'testada para calculo' in lotes_iptu.columns:
    # Removendo espaços em branco e convertendo para numérico
    lotes_iptu['testada para calculo'] = pd.to_numeric(
        lotes_iptu['testada para calculo'].str.replace(',', '.').str.strip(), 
        errors='coerce')
    lotes_iptu['testada para calculo'].fillna(0, inplace=True)

if 'geometry' in lotes_iptu.columns:
    lotes_iptu['geometry'] = lotes_iptu['geometry'].apply(lambda geom: geom.wkt if geom else None)

# Salvando o DataFrame com a coluna `geometry` em formato WKT
lotes_iptu.to_parquet(r'D:\Drive\Meu Drive\Codigos bee6\bee6_arquivos\solo_inteligente\Tratados\lotes_iptu.parquet', index=False)

C:\Users\guici\AppData\Local\Temp\ipykernel_24632\874857941.py:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  lotes_iptu['quantidade de pavimentos'].fillna(0, inplace=True)
C:\Users\guici\AppData\Local\Temp\ipykernel_24632\874857941.py:33: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a 

OSError: Cannot save file into a non-existent directory: 'D:\Drive\Meu Drive\Codigos bee6\bee6_arquivos\terraquest\Tratados'

### Trata os 2 zoneamentos diferentes

In [ ]:
# Trata os 2 zoneamentos 

# Dicionário de renomeação para o dataframe zoneamento_completo (Lei 18177)
dict_renomeio_zoneamento = {
    'areaM2': 'area_m2_lei18177','zl_zona': 'zona_lei18177',
    'zl_txt_zon': 'descr_zona_lei18177','zl_obs': 'obs_lei18177',
    'zl_tip_lei': 'tipo_lei18177','zl_ano_lei': 'ano_lei18177',
    'zl_num_lei': 'numero_lei18177','zl_id': 'id_zona_lei18177',
    'geometry': 'geometry' }

# Dicionário de renomeação para o dataframe mapa1 (Lei 16402)
dict_renomeio_mapa1 = {
    'ZONA': 'zona_lei16402','a': 'area_m2_lei16402',
    'geometry': 'geometry','Veto': 'veto_lei16402'}

lei18 = zoneamento_completo.rename(columns=dict_renomeio_zoneamento)
lei18.drop(columns=['obs_lei18177', 'tipo_lei18177', 'ano_lei18177', 'numero_lei18177'], inplace=True)
lei16 = mapa1.rename(columns=dict_renomeio_mapa1)
lei16.drop(columns=['veto_lei16402'], inplace=True)

if 'geometry' in lei16.columns:
    lei16['geometry'] = lei16['geometry'].apply(lambda geom: geom.wkt if geom else None)
if 'geometry' in lei18.columns:
    lei18['geometry'] = lei18['geometry'].apply(lambda geom: geom.wkt if geom else None)

lei18.to_parquet(r'D:\Drive\Codigos bee6\bee6_arquivos\terraquest\Tratados\lei18.parquet', index=False)

KeyError: "['obs_lei18177', 'tipo_lei18177', 'ano_lei18177', 'numero_lei18177'] not found in axis"

In [ ]:
# Cria o dataframe de acordo com a tabela Quadro 3 referente a lei 16
data = [
    ["ZEU", "ZEU", 0.5, 1, 4, 0.85, 0.70, "NA", "NA", "NA", "3 (j)", 20],
    ["ZEU", "ZEUa", "NA", 1, 2, 0.70, 0.50, 28, "NA", "NA", "3 (j)", 40],
    ["ZEUP", "ZEUP (b)", 0.5, 1, 2, 0.85, 0.70, 28, "NA", "NA", "3 (j)", "NA"],
    ["ZEUP", "ZEUPa (c)", "NA", 1, 1, 0.70, 0.50, 28, "NA", "NA", "3 (j)", "NA"],
    ["ZEM", "ZEM", 0.5, 1, "2 (d)", 0.85, 0.70, 28, "NA", "NA", "3 (j)", 20],
    ["ZEM", "ZEMP", 0.5, 1, "2 (e)", 0.85, 0.70, 28, "NA", "NA", "3 (j)", 40],
    ["ZC", "ZC", 0.3, 1, 2, 0.85, 0.70, 48, 5, "NA", "3 (j)", "NA"],
    ["ZC", "ZCa", "NA", 1, 1, 0.70, 0.70, 20, 5, "NA", "3 (j)", "NA"],
    ["ZC", "ZC-ZEIS", 0.5, 1, 2, 0.85, 0.70, "NA", 5, "NA", "3 (j)", "NA"],
    ["ZCOR", "ZCOR-1", 0.05, 1, 1, 0.50, 0.50, 10, 5, "NA", "3 (j)", "NA"],
    ["ZCOR", "ZCOR-2", 0.05, 1, 1, 0.50, 0.50, 10, 5, "NA", "3 (j)", "NA"],
    ["ZCOR", "ZCOR-3", 0.05, 1, 1, 0.50, 0.50, 10, 5, "NA", "3 (j)", "NA"],
    ["ZCOR", "ZCORa", "NA", 1, 1, 0.50, 0.50, 10, 5, "NA", "3 (j)", "NA"],
    ["ZM", "ZM", 0.3, 1, 2, 0.85, 0.70, 28, 5, "NA", "3 (j)", "NA"],
    ["ZM", "ZMa", "NA", 1, 1, 0.70, 0.50, 15, 5, "NA", "3 (j)", "NA"],
    ["ZM", "ZMIS", 0.3, 1, 2, 0.85, 0.70, 28, 5, "NA", "3 (j)", "NA"],
    ["ZM", "ZMISa", "NA", 1, 1, 0.70, 0.50, 15, 5, "NA", "3 (j)", "NA"],
    ["ZEIS", "ZEIS-1", 0.5, 1, "2.5 (f)", 0.85, 0.70, "NA", 5, "NA", "3 (j)", "NA"],
    ["ZEIS", "ZEIS-2", 0.5, 1, "4 (f)", 0.85, 0.70, "NA", 5, "NA", "3 (j)", "NA"],
    ["ZEIS", "ZEIS-3", 0.5, 1, "4 (g)", 0.85, 0.70, "NA", 5, "NA", "3 (j)", "NA"],
    ["ZEIS", "ZEIS-4", "NA", 1, "2 (h)", 0.70, 0.50, "NA", 5, "NA", "3 (j)", "NA"],
    ["ZEIS", "ZEIS-5", 0.5, 1, "4 (f)", 0.85, 0.70, "NA", 5, "NA", "3 (j)", "NA"],
    ["ZDE", "ZDE-1", 0.5, 1, 2, 0.70, 0.70, 28, 5, "NA", "3 (j)", "NA"],
    ["ZDE", "ZDE-2", 0.5, 1, 2, 0.70, 0.50, 28, 5, 3, "3 (j)", "NA"],
    ["ZPI", "ZPI-1", 0.5, 1, 1.5, 0.70, 0.70, 28, 5, 3, "3 (j)", "NA"],
    ["ZPI", "ZPI-2", "NA", 1, 1.5, 0.50, 0.30, 28, 5, 3, "3 (j)", "NA"],
    ["ZPR", "ZPR", 0.05, 1, 1, 0.50, 0.50, 10, 5, "NA", 3, "NA"],
    ["ZER", "ZER-1", 0.05, 1, 1, 0.50, 0.50, 10, 5, "NA", 3, "NA"],
    ["ZER", "ZER-2", 0.05, 1, 1, 0.50, 0.50, 10, 5, "NA", 3, "NA"],
    ["ZER", "ZERa", "NA", 1, 1, 0.50, 0.50, 10, 5, "NA", 3, "NA"],
    ["ZPDS", "ZPDS", "NA", 1, 1, 0.35, 0.25, 20, 5, "NA", 3, "NA"],
    ["ZPDS", "ZPDSr", "NA", 0.2, 0.2, 0.20, 0.15, 10, 5, "NA", 3, "NA"],
    ["ZEPAM", "ZEPAM", "NA", 0.1, 0.1, 0.10, 0.10, 10, 5, "NA", 3, "NA"],]

# Create a DataFrame
columns = [
    "TIPO DE ZONA", "ZONA", "C.A. mínimo", "C.A. básico", "C.A. máximo",
    "T.O. para lotes até 500 metros²", "T.O. para lotes igual ou superior a 500 metros²",
    "Gabarito de altura máxima (m)", "Recuo Frente (m)", "Recuo Fundos e Laterais (m)",
    "Altura Edificação Superior a 10 m", "Cota parte máxima de terreno por unidade (m²)"
]

df = pd.DataFrame(data, columns=columns)
df = df.astype(str)
# Appenda o df na lei16

#lei16.to_parquet(r'D:\Drive\Codigos bee6\bee6_arquivos\terraquest\Tratados\lei16.parquet', index=False)

### Trata Mancha Inundacao e Manancial completo

In [5]:
# Trata mancha inundacao

# Dicionário de renomeação para as colunas
dict_renomeio_inundacao = {
    'm100_profu': 'profundidade_100_anos','m100_inund': 'indice_inundacao_100_anos',
    'm100_eleva': 'elevacao_100_anos','m100_area': 'area_inundacao_100_anos',
    'm100_nome': 'nome_area_100_anos','m100_retor': 'periodo_retorno_100_anos',
    'm25_area': 'area_inundacao_25_anos','m25_nome': 'nome_area_25_anos',
    'm25_retor': 'periodo_retorno_25_anos','m25_profu': 'profundidade_25_anos',
    'm25_inund': 'indice_inundacao_25_anos','m25_eleva': 'elevacao_25_anos',
    'm5_inund': 'indice_inundacao_5_anos','m5_retor': 'periodo_retorno_5_anos',
    'm5_profu': 'profundidade_5_anos','m5_eleva': 'elevacao_5_anos',
    'm5_area': 'area_inundacao_5_anos','m5_nome': 'nome_area_5_anos'}

# Aplicando a renomeação ao dataframe mancha_inundacao
mancha_inundacao = mancha_inundacao.rename(columns=dict_renomeio_inundacao)

# Lista com a nova ordem das colunas
nova_ordem_colunas = [
    'geometry', 
    'area_inundacao_5_anos', 'nome_area_5_anos', 'periodo_retorno_5_anos', 'profundidade_5_anos', 'indice_inundacao_5_anos', 'elevacao_5_anos',
    'area_inundacao_25_anos', 'nome_area_25_anos', 'periodo_retorno_25_anos', 'profundidade_25_anos', 'indice_inundacao_25_anos', 'elevacao_25_anos',
    'area_inundacao_100_anos', 'nome_area_100_anos', 'periodo_retorno_100_anos', 'profundidade_100_anos', 'indice_inundacao_100_anos', 'elevacao_100_anos']

# Reordenando as colunas no dataframe
mancha_inundacao = mancha_inundacao[nova_ordem_colunas]

# Trata a manancial_completo

# Dicionário de renomeação para as colunas
dict_renomeio_manancial = {
    'mr_perimet': 'perimetro_manancial','mr_bacia': 'nome_bacia_hidrografica',
    'mr_area': 'area_total_manancial','mr_aod': 'autorizacao_ocupacao_desenvolvimento',
    'mr_classe': 'classe_manancial','mr_sgclass': 'subclasse_manancial',
    'mr_areaesp': 'area_especial_protecao','geometry': 'geometry'  }

# Aplicando a renomeação ao dataframe manancial_completo
manancial_completo = manancial_completo.rename(columns=dict_renomeio_manancial)

if 'geometry' in manancial_completo.columns:
    manancial_completo['geometry'] = manancial_completo['geometry'].apply(lambda geom: geom.wkt if geom else None)
if 'geometry' in mancha_inundacao.columns:
    mancha_inundacao['geometry'] = mancha_inundacao['geometry'].apply(lambda geom: geom.wkt if geom else None)

mancha_inundacao.to_parquet(r'D:\Drive\Codigos bee6\bee6_arquivos\terraquest\Tratados\mancha_inundacao.parquet', index=False)
manancial_completo.to_parquet(r'D:\Drive\Codigos bee6\bee6_arquivos\terraquest\Tratados\manancial_completo.parquet', index=False)

C:\Users\guici\AppData\Local\Temp\ipykernel_22432\3237803135.py:41: UserWarning: Geometry column does not contain geometry.
  manancial_completo['geometry'] = manancial_completo['geometry'].apply(lambda geom: geom.wkt if geom else None)
C:\Users\guici\AppData\Local\Temp\ipykernel_22432\3237803135.py:43: UserWarning: Geometry column does not contain geometry.
  mancha_inundacao['geometry'] = mancha_inundacao['geometry'].apply(lambda geom: geom.wkt if geom else None)


### Trata Risco Geologico e uso Predominante

In [6]:
# Dicionário de renomeação para risco_geologico
dict_renomeio_risco = {
    'rg_risco': 'tipo_risco_geologico','rg_sigla': 'sigla_risco',
    'rg_setor': 'setor_risco','rg_grau': 'grau_risco',
    'rg_process': 'processo_risco','rg_dtvisto': 'data_vistoria_risco',
    'rg_moradia': 'indicador_moradia_risco','geometry': 'geometry'}

# Aplicando a renomeação ao dataframe risco_geologico
risco_geologico = risco_geologico.rename(columns=dict_renomeio_risco)

# Dicionário de renomeação para uso_predominante_gdf
dict_renomeio_uso = {
    'qt_area_quadra': 'area_quadra',
    'qt_area_construida': 'area_construida',
    'uso_pred_simples': 'uso_predominante_simples',
    'uso_pred_60': 'uso_predominante_60',
    'geometry': 'geometry'  }

# Removendo as colunas 'sq', 'setor', 'quadra'
uso_predominante_gdf = uso_predominante_gdf.drop(columns=['sq', 'setor', 'quadra'])

# Aplicando a renomeação ao dataframe uso_predominante_gdf
uso_predominante_gdf = uso_predominante_gdf.rename(columns=dict_renomeio_uso)

if 'geometry' in uso_predominante_gdf.columns:
    uso_predominante_gdf['geometry'] = uso_predominante_gdf['geometry'].apply(lambda geom: geom.wkt if geom else None)
if 'geometry' in risco_geologico.columns:
    risco_geologico['geometry'] = risco_geologico['geometry'].apply(lambda geom: geom.wkt if geom else None)

uso_predominante_gdf.to_parquet(r'D:\Drive\Codigos bee6\bee6_arquivos\terraquest\Tratados\uso_predominante.parquet', index=False)
risco_geologico.to_parquet(r'D:\Drive\Codigos bee6\bee6_arquivos\terraquest\Tratados\risco_geologico.parquet', index=False)

C:\Users\guici\AppData\Local\Temp\ipykernel_22432\3215321914.py:26: UserWarning: Geometry column does not contain geometry.
  uso_predominante_gdf['geometry'] = uso_predominante_gdf['geometry'].apply(lambda geom: geom.wkt if geom else None)
C:\Users\guici\AppData\Local\Temp\ipykernel_22432\3215321914.py:28: UserWarning: Geometry column does not contain geometry.
  risco_geologico['geometry'] = risco_geologico['geometry'].apply(lambda geom: geom.wkt if geom else None)


# Parte Antiga Angelo

### Dados

In [7]:
iptu = pd.read_parquet('D:\Drive\Colab Notebooks\Vscode\Duplify\Arquivos\Iptu_Final_2023.parquet')

### Shapefiles iniciais

In [8]:
# Para cada objeto geográfico (geometry), extraio Lat_Long dos centróides dos polígonos
base_path = 'D:\Drive\Colab Notebooks\Vscode\Duplify\Arquivos\Qgis\SIRGAS_SHP_'
names = ['quadraMDSF', 'distrito', 'estacaometro_point', 'estacaotrem', 'pontoonibus', 'terminal_onibus']

for name in names:
    file_path = f"{base_path}{name}.shp"
    gdf = gpd.read_file(file_path)
    gdf = gdf.set_crs("EPSG:31983")
    
    # Check if geometry type is Polygon and extract centroid if true
    if gdf.geometry.iloc[0].geom_type == 'Polygon':
        # Calculate centroids using the projected CRS
        gdf_projected = gdf.copy()
        gdf_projected['centroid'] = gdf_projected.geometry.centroid
        gdf['centroid'] = gdf_projected['centroid'].to_crs("EPSG:4326")  # Convert back to geographic CRS
        gdf['latitude'] = gdf['centroid'].apply(lambda x: x.y).round(6)
        gdf['longitude'] = gdf['centroid'].apply(lambda x: x.x).round(6)
        
    # If geometry type is Point, directly extract latitude and longitude
    elif gdf.geometry.iloc[0].geom_type == 'Point':
        gdf = gdf.to_crs("EPSG:4326")  # Reproject to geographic CRS before extracting lat-lon
        gdf['latitude'] = gdf.geometry.apply(lambda x: x.y).round(6)
        gdf['longitude'] = gdf.geometry.apply(lambda x: x.x).round(6)
    
    # Create Lat_Long column
    gdf['Lat_Long'] = gdf['latitude'].astype(str) + ", " + gdf['longitude'].astype(str)
    gdf.drop(columns=['latitude', 'longitude', 'centroid'], inplace=True, errors='ignore')

    # Assign processed dataframe to a variable named after the file
    exec(f"{name} = gdf")

### Preparação dos dados Setor_Quadra_Lote

### Cria a coluna Setor_Quadra para extração de Lat_Long

In [9]:
# Cria o setor-quadra para que seja extraído o lat_long
def extract_setor_quadra(df):
    df['Setor_Quadra'] = df['Setor_Quadra_Lote'].str[:6]
    cols = df.columns.tolist()
    cols.insert(1, cols.pop(cols.index('Setor_Quadra')))
    return df[cols]

iptu = extract_setor_quadra(iptu)

# Deleta oq nao é relevante no momento
iptu.drop(columns = ['Ano','Codlog_Imovel','Logradouro','Numero','Cep'] , inplace = True)

### Atribui Lat_Long para cada Setor_Quadra

In [10]:
# Adiciona as coordenadas do centróide de cada polígono de quadra. 
# É mais fácil extrair lat long de cada quadra ao inves de cada lote devido ao volume de processamento
# e pelo fato da maioria das variaveis geograficas terem métricas similares considerando a localizacao da quadra apenas
quadraMDSF['Setor_Quadra'] = quadraMDSF['qd_setor'] + quadraMDSF['qd_fiscal']
quadra = quadraMDSF[['Setor_Quadra','Lat_Long']]
quadra.drop_duplicates(subset = 'Setor_Quadra', inplace = True)
iptu = iptu.merge(quadra, how = 'left', on = 'Setor_Quadra')

# Preciso criar um dataframe apenas com Lat_Long de cada setor quadra, já que se eu mantiver pra cada lote, 
# o código terá q fazer o mesmo calculo milhoes de vezes
iptu_distancia = iptu[['Setor_Quadra','Lat_Long']]
iptu_distancia.drop_duplicates(inplace = True)

C:\Users\Max Power\AppData\Local\Temp\ipykernel_19368\1667798296.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  quadra.drop_duplicates(subset = 'Setor_Quadra', inplace = True)
C:\Users\Max Power\AppData\Local\Temp\ipykernel_19368\1667798296.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  iptu_distancia.drop_duplicates(inplace = True)


### Calculos de distância

### Define funcao que separa Lat_Long e converte para float para fins de analise

In [11]:
# Funcao que separa Lat de Long já convertido pra float, para ser aplicado mais a frente
def split_lat_long_to_float(df):
    if 'Lat_Long' not in df.columns:
        print("The dataframe does not have a 'Lat_Long' column.")
        return df
    df['Lat'] = df['Lat_Long'].apply(lambda x: x.split(',')[0] if isinstance(x, str) else np.nan)
    df['Long'] = df['Lat_Long'].apply(lambda x: x.split(',')[1] if isinstance(x, str) else np.nan)
    def safe_float_conversion(value):
        try:
            return float(value)
        except:
            return np.nan
    df['Lat'] = df['Lat'].apply(safe_float_conversion)
    df['Long'] = df['Long'].apply(safe_float_conversion)
    
    return df

# Aplica para todos os dataframes
iptu_distancia = split_lat_long_to_float(iptu_distancia)
estacaometro_point = split_lat_long_to_float(estacaometro_point)
estacaotrem = split_lat_long_to_float(estacaotrem)
pontoonibus = split_lat_long_to_float(pontoonibus)
terminal_onibus = split_lat_long_to_float(terminal_onibus)
idh_geo = split_lat_long_to_float(idh_geo)

# Dropa todos os Setor_Quadra e udh com lat long nulo
iptu_distancia = iptu_distancia[np.isfinite(iptu_distancia['Lat']) & np.isfinite(iptu_distancia['Long'])]
idh_geo = idh_geo[np.isfinite(idh_geo['Lat']) & np.isfinite(idh_geo['Long'])]

C:\Users\Max Power\AppData\Local\Temp\ipykernel_19368\3824560273.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Lat'] = df['Lat_Long'].apply(lambda x: x.split(',')[0] if isinstance(x, str) else np.nan)
C:\Users\Max Power\AppData\Local\Temp\ipykernel_19368\3824560273.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Long'] = df['Lat_Long'].apply(lambda x: x.split(',')[1] if isinstance(x, str) else np.nan)
C:\Users\Max Power\AppData\Local\Temp\ipykernel_19368\3824560273.py:13: SettingWithCopyWa

### Converte Lat Long em listas de trees para calculo de distancia e retorna os valores mais proximos

In [12]:
# Cria listas de coordenadas a partir de todos os dataframes com Lat Long
def create_kdtree_from_dataframe(df):
    coords = list(zip(df['Lat'], df['Long']))
    return KDTree(coords)

# Aplica
estacaometro_point_tree = create_kdtree_from_dataframe(estacaometro_point)
estacaotrem_tree = create_kdtree_from_dataframe(estacaotrem)
pontoonibus_tree = create_kdtree_from_dataframe(pontoonibus)
terminal_onibus_tree = create_kdtree_from_dataframe(terminal_onibus)
idh_geo_tree = create_kdtree_from_dataframe(idh_geo)

# Faz o calculo do Lat Long mais proximo de cada setor quadra para cada uma das tabelas
# O retorno será uma lista com a Latitude e Longitude dos modais de transporte mais próximos, ainda não é a distancia em metros
def nearest_latlong(row, tree, target_df):
    _, index = tree.query([(row['Lat'], row['Long'])])
    return f"{target_df.iloc[index[0]]['Lat']},{target_df.iloc[index[0]]['Long']}"

suffixes = ['Metro', 'Trem', 'Ponto_Onibus', 'Terminal', 'Udh']
trees = [estacaometro_point_tree, estacaotrem_tree, pontoonibus_tree, terminal_onibus_tree, idh_geo_tree]
dfs = [estacaometro_point, estacaotrem, pontoonibus, terminal_onibus, idh_geo]

for tree, df, suffix in zip(trees, dfs, suffixes):
    col_name = f"Lat_Long_{suffix}"
    iptu_distancia[col_name] = iptu_distancia.apply(nearest_latlong, args=(tree, df), axis=1)

### Converte as coordenadas de cada modal de transporte proximo do codigo anterior em metros

In [13]:
# Na funcao anterior, o valor retornado é apenas o da Latitude e Longitude de cada modal, preciso converter para metros

# Define a function euclidean distance (in meters)
def euclidean_distance(lat1, lon1, lat2, lon2):
    meters_per_degree_latitude = 111 * 1000
    meters_per_degree_longitude = 111 * 1000 * math.cos(math.radians(lat1))  # using lat1 is an approximation
    x = (lon2 - lon1) * meters_per_degree_longitude
    y = (lat2 - lat1) * meters_per_degree_latitude
    
    return math.sqrt(x**2 + y**2)

# Define a function to compute distance from concatenated columns
def compute_distance_from_concatenated_cols(row, lat_col, long_col, concatenated_col):
    # Extract the Lat and Long values from the concatenated column
    lat2, long2 = map(float, row[concatenated_col].split(","))
    return euclidean_distance(row[lat_col], row[long_col], lat2, long2)

suffixes = ["Metro", "Trem", "Ponto_Onibus", "Terminal", "Udh"]
for suffix in suffixes:
    col_name = f"Distancia_{suffix}"
    iptu_distancia[col_name] = iptu_distancia.apply(lambda row: compute_distance_from_concatenated_cols(row, 'Lat', 'Long', f"Lat_Long_{suffix}"), axis=1)

### Normalização dos dataframes

### Readiciona, para cada Setor_Quadra, as respectivas distancias dos modais na base geral de IPTU

In [14]:
# Isso é feito para que eu nao tenha que buscar para cada Lote a distancia proxima dos modais de transporte e UDH, isso levaria mto tempo 
# e nao agregaria mta informação granular na analise

iptu_distancia = iptu_distancia[['Setor_Quadra','Lat_Long_Udh','Distancia_Metro','Distancia_Trem','Distancia_Ponto_Onibus','Distancia_Terminal']]
iptu = iptu.merge(iptu_distancia, how = 'left', on = 'Setor_Quadra')

### Normaliza os valores da tabela IPTU via MinMax

In [15]:
# Columns to scale
columns_to_scale = [
    "m2_Construcao",    
    "Fator_Obsolescencia",
    "Idade_Contribuinte"]

# Initialize the scaler
scaler = MinMaxScaler(feature_range=(0, 1))

# Fit the scaler to the data and transform
iptu[columns_to_scale] = scaler.fit_transform(iptu[columns_to_scale])

### Normaliza os valores da UDH com MinMax

In [16]:
# Dropa as desnecessárias e normaliza, essa normalização precisa acontecer antes do merge com 
# IPTU, já que precisa capturar toda a variancia da cidade de SP, independentemente de possuirem 
# quadras válidas ou não
# Estou excluindo as colunas que preciso normalizar de outra forma
idh.drop(['Distrito','Populacao_Total'], axis=1, inplace=True)
columns_to_exclude = ["Com_Lixo", "Com_Luz", "Com_Agua_Esgoto", "Com_Alvenaria", "Domiciliados"]
for column in idh.columns:
    if column not in columns_to_exclude and idh[column].dtype in ['int64', 'float64']:
        scaler = MinMaxScaler()
        idh[column] = scaler.fit_transform(idh[column].values.reshape(-1, 1))

# Atribui lat Long a tabela IDH
idh_geo = idh_geo[['Udh','Lat_Long']]
idh = idh.merge(idh_geo, how = 'left', on = 'Udh')

# Preciso dropar os Udhs com Lat_Long duplicados
idh = idh.drop_duplicates(subset='Lat_Long')

### Mescla IPTUs com UDHs

In [17]:
# Junta a merged_df final e separa os lat longs por Setor_Quadra
iptu = iptu.merge(idh, left_on='Lat_Long_Udh', right_on='Lat_Long', how='inner')

# Dropa todas as colunas que nao serao mais uteis
cols_to_drop = [col for col in iptu.columns if "Lat" in col or "Long" in col]
iptu.drop(columns=cols_to_drop, inplace=True)
del iptu['Udh']

# Preenche todos os valores nulos faltantes
numeric_cols_mean = iptu.select_dtypes(include=[np.number]).mean()
iptu.fillna(numeric_cols_mean, inplace=True)

### Exploração dos Dados

### Pré-Processamento machine learning

### Extrai todos os poligonos de Lote da cidade de Sao Paulo

In [ ]:
base_path = 'D:\\Drive\\Colab Notebooks\\Vscode\\Duplify\\Arquivos\\Qgis\\Lotes\\'
shapefile_names = [f for f in os.listdir(base_path) if f.endswith('.shp')]
all_data = []

for shapefile in shapefile_names:
    # Extract numerical values from the shapefile's name
    numbers = re.findall(r'\d+', shapefile)
    if not numbers:
        continue
    file_number = float(numbers[0])

    # Construct the full file path
    file_path = os.path.join(base_path, shapefile)
    
    gdf = gpd.read_file(file_path)
    gdf = gdf.set_crs("EPSG:31983")
    
    # Check if geometry type is Polygon and extract centroid if true
    if gdf.geometry.iloc[0].geom_type == 'Polygon':
        # Calculate centroids using the projected CRS
        gdf_projected = gdf.copy()
        gdf_projected['centroid'] = gdf_projected.geometry.centroid
        gdf['centroid'] = gdf_projected['centroid'].to_crs("EPSG:4326")  # Convert back to geographic CRS
        gdf['latitude'] = gdf['centroid'].apply(lambda x: x.y).round(6)
        gdf['longitude'] = gdf['centroid'].apply(lambda x: x.x).round(6)
        
    # If geometry type is Point, directly extract latitude and longitude
    elif gdf.geometry.iloc[0].geom_type == 'Point':
        gdf = gdf.to_crs("EPSG:4326")  # Reproject to geographic CRS before extracting lat-lon
        gdf['latitude'] = gdf.geometry.apply(lambda x: x.y).round(6)
        gdf['longitude'] = gdf.geometry.apply(lambda x: x.x).round(6)
    
    # Create Lat_Long column
    gdf['Lat_Long'] = gdf['latitude'].astype(str) + ", " + gdf['longitude'].astype(str)
    gdf.drop(columns=['latitude', 'longitude', 'centroid'], inplace=True, errors='ignore')

    # Append processed GeoDataFrame to all_data list
    all_data.append(gdf)

# Concatenate all GeoDataFrames in the list into a single DataFrame
all_data_df = pd.concat(all_data, ignore_index=True)

# Gera o id Unico para cada subquadra e appenda as colunas relevantes na lote
all_data_df['Setor_Quadra_Lote'] = all_data_df['lo_setor'] + all_data_df['lo_quadra'] + all_data_df['lo_lote']
all_data_df = all_data_df[['Setor_Quadra_Lote','geometry']]
lote = lote.merge(all_data_df,how = 'left', on = 'Setor_Quadra_Lote')